# Exploración del Dataset de Reseñas

Análisis exploratorio del archivo `data/raw/reviews.csv` — reseñas de clientes de Harbor House Café. El objetivo es entender la estructura, calidad y distribución de los datos antes de cualquier modelado.

## 1. Carga de librerías y datos

Importamos las librerías base (pandas, matplotlib, numpy, seaborn) y cargamos el dataset desde `data/raw/reviews.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df = pd.read_csv('../data/raw/reviews.csv')
print("✅ Dataset cargado correctamente")

## 2. Análisis exploratorio

Revisamos forma, tipos, valores faltantes, distribución de ratings y longitud de textos.

In [ ]:
print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nTipos de dato:\n{df.dtypes}")
print(f"\nValores faltantes por columna:\n{df.isnull().sum()}")
print(f"\nTotal de nulos en el dataset: {df.isna().sum().sum()}")

In [ ]:
print("Distribución de ratings:")
print(df['rating'].value_counts().sort_index().to_string())
print(f"\nMedia: {df['rating'].mean():.2f}")
print(f"Mediana: {df['rating'].median():.0f}")
print(f"Desvío estándar: {df['rating'].std():.2f}")
print(f"\nProporción de reseñas positivas (≥4): {(df['rating'] >= 4).mean():.1%}")
print(f"Proporción de negativas (≤2): {(df['rating'] <= 2).mean():.1%}")

In [ ]:
df['text_len'] = df['review_text'].str.len()
print("Estadísticas de longitud del texto (caracteres):")
print(df['text_len'].describe().to_string())
print(f"\nReview más corta ({df['text_len'].min()} chars):")
print(df.loc[df['text_len'].idxmin(), 'review_text'])
print(f"\nReview más larga ({df['text_len'].max()} chars):")
print(df.loc[df['text_len'].idxmax(), 'review_text'])

In [ ]:
print("Muestra de reseñas por cada nivel de rating:\n")
for rating in sorted(df['rating'].unique()):
    muestra = df[df['rating'] == rating].sample(1, random_state=42)
    print(f"--- Rating {rating} ---")
    print(muestra['review_text'].values[0][:250])
    print()

## 3. Insights clave

1. **Distribución muy desbalanceada**: el 86 % de las reseñas tienen rating ≥ 4 y la media es exactamente **4.50**. Esto significa que la clase minoritaria (rating ≤ 2) representa apenas un 6 % del dataset, lo que puede requerir técnicas de balanceo o métricas adecuadas (F1, ROC) si se modela sentimiento.

2. **Sin valores faltantes ni problemas de tipos**: las 500 filas están completas, los tipos son correctos (`int64` para `review_id` y `rating`, `str` para `review_text`). No hay nulos que gestionar.

3. **Textos de longitud homogénea**: las reseñas miden entre 86 y 231 caracteres (media ≈ 171, desvío ≈ 27). Esto sugiere que son frases cortas y estandarizadas, sin reseñas extremadamente largas o vacías. No hará falta truncar ni rellenar.

## 4. Acciones de limpieza

**No se requieren acciones de limpieza significativas.** Todas las columnas están completas, los tipos son consistentes, y las reseñas tienen texto válido y longitudes razonables. Sin embargo, se recomienda:

- Revisar si hay **filas duplicadas exactas** (aunque `review_id` ya es único, podría haber reseñas idénticas con distinto ID).
- Decidir el tratamiento de las **tildes y caracteres especiales** (se observa `Ã©` en lugar de `é`, posible problema de codificación al leer el CSV).
- Confirmar que `review_id` es realmente único (llave primaria lógica).

Vamos a verificar estos puntos rápidamente.

In [ ]:
print(f"review_id único: {df['review_id'].is_unique}")
print(f"Filas exactamente duplicadas (todas las columnas): {df.duplicated().sum()}")
print(f"Duplicados en review_text: {df['review_text'].duplicated().sum()}")
print(f"\nEncoding — muestras con caracteres extraños:")
raras = df[df['review_text'].str.contains(r'[Ã©Ã±Ã]', na=False)]
print(f"Filas afectadas: {len(raras)} ({len(raras)/len(df):.1%})")
print(f"\nEjemplo: {raras.iloc[0]['review_text'][:100] if len(raras) > 0 else 'Ninguno'}")